# **Automatic segmentation of the text**

This notebook enables an automatic segmentation of medieval texts (usefull if we want to segment texts that have not yet been edited). Supported languages are Castilian, Catalan, English, French, Italian, Latin, Portuguese in their medieval states.

# 1. Libraries import

In [ ]:
!pip install langid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941171 sha256=fbfaec7bfa4fd3359eeb3e07983c1090bce090716b31f006c3e8844f6e2f12cf
  Stored in directory: /root/.cache/pip/wheels/3c/bc/9d/266e27289b9019680d65d9b608c37bff1eff565b001c977ec5
Successfully built langid


In [ ]:
import sys
import os
from os.path import join
from transformers import BertTokenizer, AutoModelForTokenClassification
import re
import langid
import tqdm


# 2. Main functions

Function to **clean the text**:

In [ ]:
def remove_punctuation(text: str):
    punct = re.compile(r"[\.,;—:\?!’'«»“/\-]")
    cleaned_text = re.sub(punct, "", text)
    return cleaned_text

Function to **tokenize the text** (BERT takes 512 tokens max at once):

In [ ]:
def tokenize(text,tokens_per_example):
    words = text.split(" ")
    return [' '.join(words[i:i+tokens_per_example]) for i in range(0, len(words), tokens_per_example)]

Functions to get the **good labels** on the **good tokens** (need to reconstruct the word-tokens after BERT-tokenization):

In [ ]:
#get the labels
def get_labels_from_preds(preds):
    bert_labels = []
    for pred in preds[-1]:
        label = [idx for idx, value in enumerate(pred) if value == max(pred)][0]
        bert_labels.append(label)
    return bert_labels

In [ ]:
def get_correspondence(sent, tokenizer, verbose=False):
    out = {}
    tokenized_index = 0
    for index, word in enumerate(sent):
        # print(tokenizer.tokenize(word))
        tokenized_word = tokenizer.tokenize(word)
        if verbose:
            print(tokenized_word)
        out[index] = tuple(item for item in range(tokenized_index, tokenized_index + len(tokenized_word)))
        tokenized_index += len(tokenized_word)
    human_split_to_bert = out
    bert_split_to_human_split = {value: key for key, value in human_split_to_bert.items()}
    return human_split_to_bert, bert_split_to_human_split

In [ ]:
def unalign_labels(human_to_bert, predicted_labels, splitted_text, verbose=False):
    predicted_labels = predicted_labels[1:-1]
    if verbose:
        print(f"Prediction: {predicted_labels}")
        print(human_to_bert)
        print(splitted_text)
    realigned_list = []

    # itering on original text
    final_prediction = []
    for index, value in enumerate(splitted_text):
        predicted = human_to_bert[index]
        # if no mismatch, copy the label
        if len(predicted) == 1:
            correct_label = predicted_labels[predicted[0]]
            if verbose:
                print(f"Position {index}")
                print(predicted_labels)
                print(predicted[0])
                print(correct_label)
        # mismatch
        else:
            correct_label = [predicted_labels[predicted[n]] for n in range(len(predicted))]
            if verbose:
                print(f"predicted labels mismatch :{predicted_labels}")
                print(f"len predicted mismatch {len(predicted)}")
                print(f"Corresponding labels in prediction: {correct_label}")

            # BERT only propose a tokenization higher than ours
            if any([n == 1 for n in correct_label]):
                correct_label = 1
        final_prediction.append(correct_label)

    assert len(final_prediction) == len(splitted_text), "List mismatch"

    tokenized_sentence = " ".join(
        [element if final_prediction[index] != 1 else f"\n{element}" for index, element in enumerate(splitted_text)])
    if verbose:
        print(f'final prediction {final_prediction}')
        print(tokenized_sentence)
    return tokenized_sentence

### **Segmentation function**

It requires :
* the **.txt file** which is going to be segmented
* the path to the **segmentation model** (the one which has been trained, for example)
* the path to the **tokenization model** (by default model)
* the **number of tokens** per example
* the **name of the folder** in which the **output** file is going to be written
* the **device** (cpu/gpu)


In [ ]:
def tokenize_text(input_file:str,
                  model_path=None,
                  tokenizer_name=None,
                  remove_punct=False,
                  tok_models:dict=None,
                  corpus_limit=None,
                  output_dir=None,
                  tokens_per_example=None,
                  device="cpu",
                  verbose=False,
                  lang=None):
    """
    Performs tokenization with given model, tokenizer on given file
    """

    # get the file
    with open(input_file) as f:
        textL = f.read().splitlines()
    localText = " ".join(str(element) for element in textL)
    if corpus_limit:
        localText = localText[:round(len(localText)*corpus_limit)]
    if remove_punct:
        localText = remove_punctuation(localText)

    if not lang:
        codelang, _ = langid.classify(localText[:300])
        #  gestion of language identification issues
        if codelang == "an" or codelang == "oc" or codelang == "pt" or codelang == "gl":
            codelang = "es"
        if codelang == "eo" or codelang == "ht":
            codelang = "fr"
        if codelang == "jv":
            codelang = "it"
        print(f"Detected lang: {codelang}")
    else:
        codelang = lang

    # get the path of the model
    if model_path:
        pass
    else:
        model_path = tok_models[codelang]["model"]
        tokens_per_example = tok_models[codelang]["tokens_per_example"]
        tokenizer_name = tok_models[codelang]["tokenizer"]

    print(f"Using {model_path} model and {tokenizer_name} tokenizer.")
    new_model = AutoModelForTokenClassification.from_pretrained(model_path, num_labels=3)
    # get the path of the default tokenizer
    tokenizer = BertTokenizer.from_pretrained(tokenizer_name, max_length=tokens_per_example)
    new_model.to(device)

    # get the number of tokens per fragment to tokenize
    if not tokens_per_example:
        tokens_per_example = tok_models[codelang]["tokens_per_example"]
    # split the full input text as slices
    text = tokenize(localText, tokens_per_example)
    # prepare the data
    restruct = []
    # apply the tok process on each slice of text
    for i in tqdm.tqdm(text):
        # BERT-tok
        enco_nt_tok = tokenizer.encode(i, truncation=True, padding=True, return_tensors="pt")
        enco_nt_tok = enco_nt_tok.to(device)
        # get the predictions from the model
        predictions = new_model(enco_nt_tok)
        preds = predictions[0]
        # apply the functions
        bert_labels = get_labels_from_preds(preds)
        human_to_bert, bert_to_human = get_correspondence(i.split(), tokenizer)
        new_labels = unalign_labels(human_to_bert=human_to_bert, predicted_labels=bert_labels, splitted_text=i.split())
        tokenized = new_labels.split("\n")
        if verbose:
            print(i)
            print(new_labels)
            print(tokenized)

        # first token
        try:
            if tokenized[0] == "":
                restruct.extend(tokenized[1:])
            else:
                last_token = restruct[-1]
                restruct[-1] = f"{last_token} {tokenized[0]}"
                restruct.extend(tokenized[1:])
        # first token
        except IndexError:
            if tokenized[0] == "":
                restruct.extend(tokenized[1:])
            else:
                restruct.extend(tokenized)

    # testing if we don't loose tokens
    input_text_length = len(localText.split())
    output_text_length = len(" ".join(restruct).split())

    assert input_text_length == input_text_length, "Length of input text and tokenized text mismatch, something went wrong: " \
                                                   f"Input: {input_text_length}, output: {output_text_length}"
    print("No tokens were lost during the process.")

    try:
        os.mkdir(output_dir)
    except OSError as exception:
        pass

    # prepare the name of the output file
    if '/' in input_file:
        filename_corr = input_file.rpartition('/')[-1].split('.')[0]
    else:
        filename_corr = input_file.split('.')[0]

    output_file = join(output_dir, f'{filename_corr}-tok.txt')


    # write the file
    with open(output_file, "w") as text_file:
        text_file.write("\n".join(restruct))
        print(f"Saving to {output_file}")
    return restruct

# 3. Arguments



### Input file

**Note**: The path to the input file should be modified to get your own data.

Data on your drive : let the notebook access to the drive !



In [ ]:
#sys.path.append('/content/drive/MyDrive/data-align')
#from google.colab import drive
#drive.mount('/content/drive', force_remount=True)

In [ ]:
#input_file= '/content/drive/MyDrive/data-align/afEd-067.txt'



Alternatively, you can get the file from the repo of the workshop, by executing the two cells below:

In [ ]:
!git clone https://github.com/ProMeText/multilingual-medieval-aligner-workshop.git

Cloning into 'multilingual-medieval-aligner-workshop'...
remote: Enumerating objects: 6793, done.
remote: Counting objects: 100% (980/980), done.
remote: Compressing objects: 100% (406/406), done.
remote: Total 6793 (delta 628), reused 610 (delta 563), pack-reused 5813 (from 1)
Receiving objects: 100% (6793/6793), 365.05 MiB | 23.36 MiB/s, done.
Resolving deltas: 100% (2926/2926), done.


In [ ]:
input_file = 'multilingual-medieval-aligner-workshop/data/to-segment/Val_S.txt'

### List of other arguments

In [ ]:
model_path = "ProMeText/aquilign-multilingual-segmenter"
tokenizer_name = "google-bert/bert-base-multilingual-cased"
remove_punct = False
example_length = 100
device = 'cpu'
#device = 'cuda:0'
output_dir = ''

### Apply the function

The function produces a `XXX-tok.txt` file which is stored where the `output_dir` indicates.

In [ ]:
tokenize_text(model_path=model_path,
              tokenizer_name=tokenizer_name,
              remove_punct=remove_punct,
              input_file=input_file,
              tokens_per_example=example_length,
              device=device,
              output_dir=output_dir)

Detected lang: es
Using ProMeText/aquilign-multilingual-segmenter model and google-bert/bert-base-multilingual-cased tokenizer.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

100%|██████████| 31/31 [00:10<00:00,  2.82it/s]

No tokens were lost during the process.
Saving to Val_S-tok.txt


['¶ El primer capitło de commo se departen los pode rios del alma· ',
 'Et en quales poderios han de seer las uirtudes / · ',
 'ffolibro.¶de la segunda libro parte del primo ues que ya con el ayuda de dios acabamos la primera parte deste libro primero ',
 'en que se tracta del gouernamiento del omnen en ssi /· ',
 'Et mostramos en que deuen poner los Reyes ',
 'e los prinçipes la su feliçidat ',
 'e la su bien andança /· ',
 'Et que non los conuiene poner la su fin en riquezas ',
 'nin en poderio çiuil ',
 'nin en nuguaso tris cosas corporales nin tenporales ',
 'Mas assi commo prouamos conplidamente de suso deuen husar de todas estas cosas ',
 'assi commo de instrumentos ',
 'para ganar la feliçidat ',
 'e la bien andança · ',
 'Mas la su bien andança deuen poner en obras de pradençia ',
 'e de sabiduria segund ',
 'que tales obras son regladas ',
 'por caridat ',
 'e por el amor de dios ',
 'Ca estonçe han los Reyes ',
 'e los prinçipes la su feliçidait ',
 'e la su feliç